# RAG PDF Research Corpus System

**Version:** 1.0  
**Date:** 2025-11-21  
**Author/Maintainer:** Research Corpus Organizer  
**Based on:** RAG_PDF_System_Spec_v2.1

---

## Overview

This notebook implements a comprehensive system for processing academic PDF research papers using:
- **LangGraph** workflows for orchestration
- **GPT-5.1 Thinking** for summarization and classification
- **FAISS** for vector indexing and RAG queries
- **3-tier hierarchical topic taxonomy** for organization

### System Capabilities

1. **Ingest** PDFs from Google Drive
2. **Parse and chunk** documents with section awareness
3. **Extract metadata** from arXiv/CrossRef/PDF sources
4. **Generate summaries** using advanced LLMs
5. **Build topic taxonomy** through clustering
6. **Classify papers** into hierarchical topics
7. **Enable RAG queries** for corpus exploration

---

## Phase 0: Environment Setup and Configuration

This phase establishes the notebook environment, installs dependencies, and configures the system.

### Step 0.2: Environment Inspection

First, let's verify the runtime environment meets our requirements.

In [ ]:
# Check Python version (require 3.10+)
import sys
print(f"Python version: {sys.version}")
print(f"Version info: {sys.version_info}")

if sys.version_info >= (3, 10):
    print("✓ Python 3.10+ requirement met")
else:
    print("✗ WARNING: Python 3.10+ required")

In [ ]:
# Check GPU/CPU availability
import os

# Try to check for GPU
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
        print(f"  CUDA version: {torch.version.cuda}")
    else:
        print("○ No GPU available (using CPU)")
except ImportError:
    print("○ PyTorch not installed yet (will check GPU after installation)")

# Display basic system info
import platform
print(f"\nSystem: {platform.system()} {platform.release()}")
print(f"Machine: {platform.machine()}")
print(f"Processor: {platform.processor()}")

In [ ]:
# Display runtime information
import os
try:
    import psutil
    
    # Memory information
    memory = psutil.virtual_memory()
    print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
    print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
    print(f"Used RAM: {memory.used / (1024**3):.2f} GB ({memory.percent}%)")
    
    # Disk space
    disk = psutil.disk_usage('/')
    print(f"\nTotal Disk: {disk.total / (1024**3):.2f} GB")
    print(f"Available Disk: {disk.free / (1024**3):.2f} GB")
    print(f"Used Disk: {disk.used / (1024**3):.2f} GB ({disk.percent}%)")
    
    # CPU information
    print(f"\nCPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count(logical=True)} logical")
except ImportError:
    print("psutil not installed yet - will be available after dependency installation")

### Step 0.3: Install Dependencies

Install all required packages for the RAG PDF Research Corpus System.

**Note:** After installation, you may need to restart the runtime if prompted.

In [ ]:
# Install all required dependencies
# This cell may take several minutes to complete

import sys

# Core dependencies with specific versions
dependencies = [
    "openai>=1.3.0",           # GPT-5.1 support
    "langgraph>=0.0.30",       # Workflow orchestration
    "langchain>=0.1.0",        # LangChain integration
    "pymupdf>=1.23.0",         # PDF parsing (fitz)
    "faiss-cpu>=1.7.4",        # Vector indexing (CPU version)
    "scikit-learn>=1.3.0",     # Clustering algorithms
    "hdbscan>=0.8.33",         # Density-based clustering
    "pandas>=2.0.0",           # Data handling
    "numpy>=1.24.0",           # Numerical operations
    "tqdm>=4.65.0",            # Progress bars
    "matplotlib>=3.7.0",       # Visualization
    "seaborn>=0.12.0",         # Statistical visualization
    "python-dateutil>=2.8.2",  # Date parsing
    "requests>=2.31.0",        # HTTP requests for APIs
    "pytesseract>=0.3.10",     # OCR (optional)
    "Pillow>=10.0.0",          # Image processing for OCR
    "pydantic>=2.0.0",         # Data validation
    "psutil>=5.9.0",           # System utilities
]

print("Installing dependencies...")
print("=" * 60)

for dep in dependencies:
    print(f"Installing {dep}...")
    !pip install -q {dep}

print("=" * 60)
print("✓ All dependencies installed successfully!")
print("\n⚠ If you see any warnings about restarting the runtime, please do so now.")
print("   After restart, skip this installation cell and continue with imports.")

### Step 0.4: Import Statements

Import all required libraries and verify they load successfully.

In [ ]:
# Standard library imports
import os
import sys
import json
import hashlib
import logging
from pathlib import Path
from datetime import datetime, date
from typing import Optional, Dict, List, Literal, Any, TypedDict
from dataclasses import dataclass, field

# Third-party imports - Core
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Third-party imports - Data validation
from pydantic import BaseModel, Field, validator

# Third-party imports - PDF processing
try:
    import fitz  # PyMuPDF
    print("✓ PyMuPDF (fitz) imported successfully")
except ImportError as e:
    print(f"✗ Error importing PyMuPDF: {e}")
    fitz = None

# Third-party imports - Vector store
try:
    import faiss
    print("✓ FAISS imported successfully")
except ImportError as e:
    print(f"✗ Error importing FAISS: {e}")
    faiss = None

# Third-party imports - ML/Clustering
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score
try:
    import hdbscan
    print("✓ HDBSCAN imported successfully")
except ImportError:
    print("○ HDBSCAN not available (optional)")
    hdbscan = None

# Third-party imports - Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Third-party imports - API and utilities
import requests
from dateutil import parser as date_parser

# Third-party imports - OpenAI
try:
    from openai import OpenAI
    print("✓ OpenAI SDK imported successfully")
except ImportError as e:
    print(f"✗ Error importing OpenAI: {e}")
    OpenAI = None

# Third-party imports - LangGraph
try:
    from langgraph.graph import StateGraph, END
    from langgraph.checkpoint.memory import MemorySaver
    print("✓ LangGraph imported successfully")
except ImportError as e:
    print(f"✗ Error importing LangGraph: {e}")
    StateGraph = None

# Third-party imports - OCR (optional)
try:
    import pytesseract
    from PIL import Image
    print("✓ OCR libraries imported successfully")
except ImportError:
    print("○ OCR libraries not available (optional)")
    pytesseract = None
    Image = None

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("\n✓ All core imports completed successfully!")

### Step 0.5: Configuration

Define the configuration schema and user-editable configuration.